In [1]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_core.tools import tool, Tool
from pydantic import BaseModel


import subprocess
import sys 
from uuid import uuid4
import json

### Build tool registry JSON from function definitions.
The LLM can search the registry, fetch and execute scripts, and push new scripts to the registry


In [2]:
from src.tool_registry import ScriptRegistry, build_registry_tools

registry = ScriptRegistry()
tools = build_registry_tools(registry)

# After reviewing a generated script, enable it explicitly:
# registry.enable_script("the-id-returned-by-write_script_tool")


In [3]:
chat = ChatOllama(
    model="qwen3.5:4b",
    temperature=0,
    max_tokens=2048,
)

SystemPrompt = """You are a voice assistant with a persistent Python tool registry.
Search with search_scripts before creating a tool. Use execute_script with an
existing enabled tool's ID and schema-compatible arguments when it fits.
If no suitable tool exists, use write_script_tool to save a new script. Give
its entry-point function typed named parameters, a return annotation, a clear
docstring, and a JSON-serializable result. script_name has no path or extension.
New scripts are pending_validation: explain that review and enabling are needed
before execution. Never claim a pending script has run. Tool failures are
returned as error objects; read them and correct the request when appropriate.
"""
tools = build_registry_tools(registry)

messages = [HumanMessage(content="Print the word Hell after 3 seconds.")]
messages_write_python_script = [HumanMessage(
    content="Create and register a reusable function that takes two numbers and returns their sum."
)]


def llm_chat(messages: list, tools: list | None = None, max_rounds: int = 8) -> list:
    available = {registered_tool.name: registered_tool for registered_tool in (tools or [])}
    model_with_tools = chat.bind_tools(list(available.values())) if available else chat
    for _ in range(max_rounds):
        result = model_with_tools.invoke([SystemMessage(content=SystemPrompt)] + messages)
        messages.append(result)
        if result.content:
            print(f"[AGENT]: {result.content}")
        if not result.tool_calls:
            return messages
        for call in result.tool_calls:
            print(f"[TOOL CALL]: {call['name']}")
            try:
                registered_tool = available[call["name"]]
                tool_result = registered_tool.invoke(call["args"])
            except Exception as exc:
                tool_result = {"error": str(exc)}
            print(f"[OBS]: {tool_result}")
            messages.append(ToolMessage(
                content=json.dumps(tool_result), tool_call_id=call["id"]
            ))
    raise RuntimeError("Agent reached its tool-call round limit.")


In [4]:
result = llm_chat(messages_write_python_script, tools)

[TOOL CALL]: write_script_tool
[OBS]: {'error': "A tool named 'add_numbers' is already registered."}
[TOOL CALL]: search_scripts
[OBS]: [{'name': 'add_numbers', 'description': 'Add two numbers and return their sum.', 'parameters': {'additionalProperties': False, 'properties': {'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'A'}, 'b': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'B'}}, 'required': ['a', 'b'], 'title': 'ScriptArguments', 'type': 'object'}, 'return_type': 'int | float', 'id': '6d376059-2bfd-46bc-a2d6-13072db5e683', 'path': 'agent_generated_scripts/add_numbers.py', 'entry_point': 'add_numbers', 'status': 'pending_validation', 'version': 1, 'sha256': '78040f4232b790714266939210d8edad8c5dcbc6bfbea4a12bb7e28d1b97a781'}]
[AGENT]: A script named `add_numbers` already exists in your registry with the exact functionality you requested:

- **Function**: Takes two numbers (int or float) and returns their sum
- **Parameters**: `a` and `b` (both i

#### OKAY, we have a primitive (but working) llm agent, now let's add the ASR model:

In [5]:
from qwen_asr import Qwen3ASRModel
import torch

model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-0.6B",
    dtype=torch.bfloat16,
    device_map="cuda:0",
    # attn_implementation="flash_attention_2",
    max_inference_batch_size=32, # Batch size limit for inference. -1 means unlimited. Smaller values can help avoid OOM.
    max_new_tokens=256, # Maximum number of tokens to generate. Set a larger value for long audio input.
)



/home/bashar/Study/Projects/AIVoiceAssistant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [6]:
## Record 5-second audio
import sounddevice as sd
from scipy.io.wavfile import write

sample_rate = 24000
duration = 5

def record_audio(sr, duration) :
    print(f"Recording audio for {duration} seconds...")
    audio = (sd.rec(
        int(duration * sr),
        samplerate=sr,
        channels=1,
        dtype="int16",
    ), sample_rate)
    sd.wait()
    return audio

In [ ]:
%time
results = model.transcribe(
    audio=audio,
    language="English"
)
print(results)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


CPU times: user 0 ns, sys: 14 μs, total: 14 μs
Wall time: 25 μs
[ASRTranscription(language='English', text='Hello, what is the capital of France?', time_stamps=None)]


### Combine audio -> ASR -> LLM -> Output

In [12]:
audio = record_audio(sr=24000, duration=10)
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content= text)]
llm_result = llm_chat(messages=messages, tools=tools)


Recording audio for 10 seconds...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[TOOL CALL]: search_scripts
[OBS]: [{'error': 'limit must be between 1 and 20.'}]
[TOOL CALL]: search_scripts
[OBS]: [{'name': 'add_numbers', 'description': 'Add two numbers and return their sum.', 'parameters': {'additionalProperties': False, 'properties': {'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'A'}, 'b': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'B'}}, 'required': ['a', 'b'], 'title': 'ScriptArguments', 'type': 'object'}, 'return_type': 'int | float', 'id': '6d376059-2bfd-46bc-a2d6-13072db5e683', 'path': 'agent_generated_scripts/add_numbers.py', 'entry_point': 'add_numbers', 'status': 'enabled', 'version': 1, 'sha256': '78040f4232b790714266939210d8edad8c5dcbc6bfbea4a12bb7e28d1b97a781'}, {'name': 'main', 'description': 'Main entry point for the script.\n\nReturns:\n    dict[str, float]: Result containing the square root of the product', 'parameters': {'additionalProperties': False, 'properties': {}, 'title': 'ScriptArguments', 'type': '

HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[AGENT]: Here are all the available Python scripts in the registry:

## Executable Scripts (Enabled):
1. **add_numbers** (`6d376059-2bfd-46bc-a2d6-13072db5e683`)
   - Description: Add two numbers and return their sum.
   - Parameters: `a` (number), `b` (number)
   - Return type: int | float

## Pending Validation Scripts (Not Executable):
2. **main** (`1eb940d3-f4e7-4c45-bcc0-90d305e1b642`)
   - Description: Main entry point for the script, returns square root of product
   - Parameters: None (empty)
   - Return type: dict[str, float]

3. **fibonacci_calculator** (`6ce1f2b1-b877-4e10-985f-ff6ac96b1a98`)
   - Description: Calculate the Fibonacci number at a given index position
   - Parameters: `index` (integer)
   - Return type: int

**Summary:** Only **1 script** is currently executable (`add_numbers`). The other 2 scripts are in `pending_validation` status and require review and enabling before they can be executed.


In [ ]:
llm_result

[HumanMessage(content='What is the secret of the universe that no one else knows, and you can tell me it right now without any hesitation, please?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The "secret of the universe" is something that has fascinated humanity for millennia, but there isn\'t one single answer that everyone agrees on. Here are a few perspectives:\n\n**From a scientific standpoint:**\n- The universe appears to be expanding at an accelerating rate, driven by dark energy\n- We\'re still searching for what constitutes dark matter and dark energy (together they make up about 95% of the universe)\n- The fundamental constants of physics seem fine-tuned for life\n\n**From a philosophical standpoint:**\n- Some suggest consciousness itself is the secret - that we are the universe experiencing itself\n- Others believe the true secret lies in understanding our place within the cosmos and finding meaning in existence\n\n**From an existential perspective:**\n-

### The Pipeline looks fairly good, we will now deal with voice verification 

In [14]:
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier

classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
signal, fs =torchaudio.load('audio_wavs/recording.wav')
print(signal.shape)
signal = signal.mean(dim=0, keepdim=True)
signal = torchaudio.functional.resample(signal, fs, 16000) # necessary for ecapa-tdnn
embeddings = classifier.encode_batch(signal)


Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


torch.Size([1, 120000])


In [15]:
from speechbrain.inference.speaker import SpeakerRecognition
verification = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="pretrained_models/spkrec-ecapa-voxceleb")
score, prediction = verification.verify_files("audio_wavs/recording.wav", "audio_wavs/recording_dif.wav") # Different Speakers
print(f"{score= }, {prediction= }")
score, prediction = verification.verify_files("audio_wavs/recording.wav", "audio_wavs/recording_sim.wav") # Same Speaker
print(f"{score= }, {prediction= }")

Fetch hyperparams.yaml: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/hyperparams.yaml'
Fetch embedding_model.ckpt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/embedding_model.ckpt'
Fetch mean_var_norm_emb.ckpt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/mean_var_norm_emb.ckpt'
Fetch classifier.ckpt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/classifier.ckpt'
Fetch label_encoder.txt: Using symlink found at '/home/bashar/Study/Projects/AIVoiceAssistant/pretrained_models/spkrec-ecapa-voxceleb/label_encoder.ckpt'
Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


score= tensor([0.2358], device='cuda:0'), prediction= tensor([False], device='cuda:0')
score= tensor([0.7795], device='cuda:0'), prediction= tensor([True], device='cuda:0')


In [16]:

import torch
import torch.nn.functional as F 


def enroll_speaker(trials=2, trial_duration=5):
    pass_threshold = 0.45
    print(f"You will be asked to speak {trials} times, each time you can say whatever you want for {trial_duration} seconds, the model will save your voiceprint afterwards")
    embeddings = []
    for i in range(trials):
        input(f"Press Enter to Start trial {i+1}...")
        print(f"Trial number {i+1} : ")
        audio, sr = record_audio(16000, trial_duration)
        audio = torch.tensor(audio, dtype=torch.float32)
        audio = audio.T
        audio = audio.mean(dim=0, keepdim=True) # in case of multiple channels, ecapa accepts mono audio with sr = 16000
        audio = torchaudio.functional.resample(audio, sr, 16000)
        embeddings.append(classifier.encode_batch(audio, normalize=False))
        if i > 0:
            similarity = F.cosine_similarity(embeddings[-1][0,:,:], embeddings[-2][0,:,:])
            assert similarity.item() > pass_threshold, "Your recorded voice is very different from the previous, try again"
    res = torch.cat(embeddings, dim=0)
    res = F.normalize(res, p=2, dim=-1)
    print(f"Finished Enrollment Procedure")
    return F.normalize(res.mean(dim=0), dim=-1)



def authorized_speaker(audio, sr, ref_dim:torch.tensor) -> bool:
    """Checks the similarity with the enrolled speaker """
    if not isinstance(audio, torch.Tensor):
        audio = torch.tensor(audio, dtype=torch.float32)
        if audio.shape[0] != 1:
            audio = audio.T
    print(audio.shape)
    audio = audio.mean(dim=0, keepdim=True)
    audio = torchaudio.functional.resample(audio, sr, 16000)
    audio_dim = classifier.encode_batch(audio, normalize=False)[0]
    audio_dim = F.normalize(audio_dim, p=2, dim=-1)
    sim = F.cosine_similarity(audio_dim, ref_dim)
    print(f"{sim=}")
    return sim.item() > 0.45




In [17]:
authorized_spk_dim = enroll_speaker(trials=3, trial_duration=5)

You will be asked to speak 3 times, each time you can say whatever you want for 5 seconds, the model will save your voiceprint afterwards
Trial number 1 : 
Recording audio for 5 seconds...
Trial number 2 : 
Recording audio for 5 seconds...
Trial number 3 : 
Recording audio for 5 seconds...
Finished Enrollment Procedure


In [18]:
audio = record_audio(16000, 5)

Recording audio for 5 seconds...


In [19]:
authorized_speaker(audio[0], audio[1], authorized_spk_dim)

torch.Size([1, 80000])
sim=tensor([0.7530], device='cuda:0')


True

### The speaker verification is working fine for now (needs further testing to check the authorization).
### The next step is to add it to the pipeline : Audio -> Speaker authorization -> ASR -> LLM -> Response

In [25]:
audio = record_audio(16000, 10)
if not authorized_speaker(audio[0], audio[1], authorized_spk_dim):
    raise Exception('Unauthorized speaker.')
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content=text)]
tools = build_registry_tools(registry)
response = llm_chat(messages=messages, tools=tools)

Recording audio for 10 seconds...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


torch.Size([1, 160000])
sim=tensor([0.7155], device='cuda:0')


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[AGENT]: Hello Shane! Welcome to Reddit Stories. It's great to have you here today. 

Dino Week is a fantastic theme - dinosaurs are such an exciting topic for storytelling! I'm ready to help you create content or scripts for your show. 

What would you like to work on today? Are you looking to:
- Create a new script about dinosaurs?
- Search for existing scripts related to dinosaurs?
- Or perhaps something else entirely?

Let me know how I can assist with your Reddit Stories episode!
